In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
#import tensorflow_addons as tfa
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.layers import LSTM,Bidirectional,GRU
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.utils import to_categorical
import datetime
import io
import itertools
# import seaborn as sns


In [4]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-07-10 09:52:49.814112: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-07-10 09:52:49.906360: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-07-10 09:52:49.906685: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [9]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from sklearn.model_selection import KFold
from sklearn.metrics import classification_report

import sys
import os
# Obtener la ruta del directorio actual
os.chdir('/home/rgadea/experimentos_software_2024/nuevas_investigaciones_alimentos_2024')
current_dir = os.getcwd()
print(current_dir)

# Construir la ruta relativa al directorio que quieres agregar
relative_dir = os.path.join(current_dir, 'mis_pkgs/')

# Agregar la ruta relativa al sys.path
sys.path.insert(0, relative_dir)

#from MIOPATIA_db import DB_management as db 


/home/rgadea/experimentos_software_2024/nuevas_investigaciones_alimentos_2024


In [10]:
numero_muestras=201
numero_clases=2
entrada=[5,6]
numero_entradas =2
numero_epochs=20000

Voy a quedarme con los 50 atunes P1 para obtener conjunto de training y validacion

In [12]:
filename = "COPIA_PANDAS/medidas_agilent_2023_y_2024_201_puntos_clasificados.hdf"
with pd.HDFStore(filename,complib="zlib",complevel=4) as hdf_db:
    pre_p_e1  = hdf_db.get('data/pollos_estado')
    pre_p_e1 = pre_p_e1.loc[pre_p_e1['Pollo'] != 0]
    # p_e =pre_p_e1.drop_duplicates(subset = ['Pollo', 'Medida'],  keep = 'last').reset_index(drop = True)
    t    = hdf_db.get('data/tabla')
    X_train=np.zeros((pre_p_e1.shape[0],numero_muestras,numero_entradas))
    y_train=np.zeros((pre_p_e1.shape[0],1))
    x=0
    for index, row in pre_p_e1.iterrows():   # El primer registro no se toma en cuenta porque es basura
        Primero = int(row['Primero'])
        Ultimo  = int(row['Ultimo'])
        estado  = int(row['Estado'])
        #print(Primero)
        #print(Ultimo)
        #print(estado)
        if numero_clases==2:
            if estado == 0 or estado== 1:
                target = 0
            else:
                target = 1
        else:
            target=estado
        pepito=np.array(t.iloc[Primero:Ultimo+1])
        # #print(pepito.shape)
        X_train[x]=pepito[:,2]*pepito[:,entrada]
        #print(X_train[x][0:4,:])       
        y_train[x]=target
        y_train_to_categorical = to_categorical(y_train)
        x=x+1



X_train_filtrado = X_train
#y_train_filtrado = y_train
y_train_filtrado = y_train_to_categorical


scaler = MinMaxScaler(feature_range=(0, 1))
#scaler = StandardScaler()



#data1=np.concatenate((X_train_filtrado,X_test_filtrado1),axis=0) 

data_2d = X_train_filtrado.reshape(-1, X_train_filtrado.shape[-1])
normalized_data_2d = scaler.fit_transform(data_2d)



X_train_Normalizado=normalized_data_2d.reshape(X_train_filtrado.shape)
y_train_Normalizado=y_train_filtrado # los valores ya estaban normalizados
print(data_2d.shape)
print(X_train_Normalizado.shape)
print(y_train_Normalizado.shape)

inputs=X_train_Normalizado.reshape(X_train_Normalizado.shape[0],-1)
targets=y_train_Normalizado

print(inputs.shape)
print(targets.shape)



(38994, 2)
(194, 201, 2)
(194, 2)
(194, 402)
(194, 2)


Vamos a hacer los conjuntos de entrenamiento validacion y test

In [13]:
factor_aprendizaje=0.001
dimension_LSTM=200
dimension_dense1=50
dimension_dense2=20
algoritmo='rmsprop'
supermax=8*4
lossfunction='categorical_crossentropy'
def create_model():

    model = Sequential()
    model.add(Bidirectional(GRU(dimension_LSTM, return_sequences=True,recurrent_regularizer='L2',input_shape=(numero_muestras, numero_entradas))))
    model.add(Flatten())  
    #model.add(GRU(50, return_sequences=True))
    #model.add(GRU(50, return_sequences=False, recurrent_regularizer='L2'))
    model.add(Dense(dimension_dense1, activation='tanh', activity_regularizer='L2'))
    model.add(Dense(dimension_dense2, activation='tanh'))
    model.add(Dense(numero_clases, activation='softmax'))
    model.compile(loss=lossfunction, optimizer=algoritmo, metrics=['accuracy',
                              tf.keras.metrics.Recall(class_id=0),
                              tf.keras.metrics.Recall(class_id=1) #,
                              #tfa.metrics.F1Score(num_classes=numero_clases,average='macro', threshold=0.5)
                              ])
    model.optimizer.lr=(factor_aprendizaje)
    return model



In [14]:

experimento="LOMOS_Agilent_5clases_GRU1_{}_dense1_{}_dense2_{}_loss_{}_lr_{}_algoritmo_{}".format(dimension_LSTM,dimension_dense1,dimension_dense2,lossfunction,factor_aprendizaje,algoritmo)
logdir="./logs/defs/leavekout/{}_{}".format(experimento,datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback=tf.keras.callbacks.TensorBoard(log_dir=logdir, histogram_freq=1)
file_writer_cm = tf.summary.create_file_writer(logdir + '/cm')


2024-07-10 09:59:48.954809: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-07-10 09:59:48.955574: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-07-10 09:59:48.955958: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [16]:
if numero_clases==2:
    class_names=['Buenos', 'Malos']
else:
    class_names=['A', 'B+', 'B', 'B-','C']

In [17]:
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.95,
    patience=1000,
    min_lr=0.0001
)
early_stop=tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=0, patience=2000, verbose=2, mode='auto', baseline=None, restore_best_weights=True)
# Define the K-fold Cross Validator
kfold = KFold(n_splits=10, shuffle=True)
# Define per-fold score containers
acc_per_fold = []
loss_per_fold = []
sensibilidad_YAKE_per_fold=[]
sensibilidad_no_YAKE_per_fold=[]
# K-fold Cross Validation model evaluation
fold_no = 1
for train, test in kfold.split(inputs, targets):
    model=create_model()
     # Generate a print
    print('------------------------------------------------------------------------')
    print(f'Training for fold {fold_no} ...')
    inputs_good=inputs.reshape(X_train_filtrado.shape)
    # Fit data to model
    history = model.fit(inputs_good[train], targets[train],
              batch_size=20,
              epochs=numero_epochs,
              callbacks=[early_stop,lr_callback, tensorboard_callback],
              validation_data=(inputs_good[test],targets[test])
              )
    if numero_clases==2:
        target_names = ['Buenos', 'Malos']
    else:   
        target_names = ['A', 'B+', 'B', 'B-','C']
    y_pred = model.predict(inputs_good[test])
    y_pred2=np.argmax(y_pred,axis=1)
    y_test_def2=np.argmax(targets[test],axis=1)
    print(classification_report(y_test_def2, y_pred2, target_names=target_names, digits=4))
    # Generate generalization metrics
    scores = model.evaluate(inputs_good[test], targets[test], verbose=0)
    print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {scores[0]}; {model.metrics_names[1]} of {scores[1]*100}%')
    print(scores[2])
    print(scores[3])
   # print(scores[4])
    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])
    sensibilidad_YAKE_per_fold.append(scores[2] * 100)
    sensibilidad_no_YAKE_per_fold.append(scores[3] * 100)
    # Increase fold number
    fold_no = fold_no + 1

/home/rgadea/experimentos_software_2024/miniconda3/envs/tensorflow_gpu_bestial/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


------------------------------------------------------------------------
Training for fold 1 ...
Epoch 1/20000


2024-07-10 10:00:04.507218: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.4442 - loss: 4.5244 - recall: 0.5694 - recall_1: 0.3091

2024-07-10 10:00:05.978866: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 964800000 exceeds 10% of free system memory.


...

9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 167ms/step - accuracy: 0.5340 - loss: 5.4777 - recall_14: 0.5439 - recall_15: 0.5053 - val_accuracy: 0.4737 - val_loss: 3.3554 - val_recall_14: 0.2500 - val_recall_15: 0.6364 - learning_rate: 0.0010
Epoch 2/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 97ms/step - accuracy: 0.5612 - loss: 3.1710 - recall_14: 0.7256 - recall_15: 0.3180 - val_accuracy: 0.4211 - val_loss: 2.7260 - val_recall_14: 1.0000 - val_recall_15: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 102ms/step - accuracy: 0.5253 - loss: 2.5050 - recall_14: 0.8927 - recall_15: 0.0593 - val_accuracy: 0.4211 - val_loss: 2.1477 - val_recall_14: 1.0000 - val_recall_15: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.6065 - loss: 1.9828 - recall_14: 0.9486 - recall_15: 0.0756 - val_accuracy: 0.5263 - val_loss: 1.7112 - val_recall_14: 0.8750 - val_recall_15: 0.2727 - learning_rate: 0.0010
Epoch 5/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 1

/home/rgadea/experimentos_software_2024/miniconda3/envs/tensorflow_gpu_bestial/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 146ms/step - accuracy: 0.5724 - loss: 5.0280 - recall_16: 0.7357 - recall_17: 0.3284 - val_accuracy: 0.3158 - val_loss: 3.4957 - val_recall_16: 0.0000e+00 - val_recall_17: 1.0000 - learning_rate: 0.0010
Epoch 2/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.4591 - loss: 3.1497 - recall_16: 0.5028 - recall_17: 0.3952 - val_accuracy: 0.6842 - val_loss: 2.5114 - val_recall_16: 1.0000 - val_recall_17: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.4720 - loss: 2.4935 - recall_16: 0.6535 - recall_17: 0.2903 - val_accuracy: 0.5789 - val_loss: 2.0445 - val_recall_16: 0.8462 - val_recall_17: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.5492 - loss: 1.9327 - recall_16: 0.7912 - recall_17: 0.2413 - val_accuracy: 0.6842 - val_loss: 1.5724 - val_recall_16: 1.0000 - val_recall_17: 0.0000e+00 - learning_rate: 0.0010
Epoch 5/20000
9/9 ━━━━━━━━━━━━━━━━

/home/rgadea/experimentos_software_2024/miniconda3/envs/tensorflow_gpu_bestial/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/rgadea/experimentos_software_2024/miniconda3/envs/tensorflow_gpu_bestial/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/rgadea/experimentos_software_2024/miniconda3/envs/tensorflow_gpu_bestial/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in label

Epoch 1/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - accuracy: 0.5210 - loss: 4.8043 - recall_18: 0.6878 - recall_19: 0.2940 - val_accuracy: 0.4737 - val_loss: 3.2790 - val_recall_18: 0.8889 - val_recall_19: 0.1000 - learning_rate: 0.0010
Epoch 2/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.5120 - loss: 3.0885 - recall_18: 0.7409 - recall_19: 0.2021 - val_accuracy: 0.5263 - val_loss: 2.5554 - val_recall_18: 0.0000e+00 - val_recall_19: 1.0000 - learning_rate: 0.0010
Epoch 3/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.4843 - loss: 2.4367 - recall_18: 0.5465 - recall_19: 0.3840 - val_accuracy: 0.4737 - val_loss: 2.0275 - val_recall_18: 1.0000 - val_recall_19: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/20000
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.5331 - loss: 1.9173 - recall_18: 0.8816 - recall_19: 0.1026 - val_accuracy: 0.4737 - val_loss: 1.6277 - val_recall_18: 1.0000 - val_recall_19: 0.0000e+00 - learning_rate: 0.0010
Epoch 5/20000
9/9 ━━━━━━

In [18]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(acc_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Loss: {loss_per_fold[i]} - Accuracy: {acc_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Accuracy: {np.mean(acc_per_fold)} (+- {np.std(acc_per_fold)})')
print(f'> Loss: {np.mean(loss_per_fold)}')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Loss: 0.6678581237792969 - Accuracy: 75.0%
------------------------------------------------------------------------
> Fold 2 - Loss: 3.2928223609924316 - Accuracy: 44.999998807907104%
------------------------------------------------------------------------
> Fold 3 - Loss: 0.7720484733581543 - Accuracy: 80.0000011920929%
------------------------------------------------------------------------
> Fold 4 - Loss: 3.3535072803497314 - Accuracy: 40.00000059604645%
------------------------------------------------------------------------
> Fold 5 - Loss: 3.251016855239868 - Accuracy: 57.894736528396606%
------------------------------------------------------------------------
> Fold 6 - Loss: 3.4764716625213623 - Accuracy: 31.578946113586426%
------------------------------------------------------------------------
> Fold 7 - 

In [19]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(sensibilidad_YAKE_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Sensibilidad YAKE: {sensibilidad_YAKE_per_fold[i]} - Sensibilidad NOYAKE: {sensibilidad_no_YAKE_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Sensibilidad YAKE: {np.mean(sensibilidad_YAKE_per_fold)} (+- {np.std(sensibilidad_YAKE_per_fold)})')
print(f'> Sensibilidad NOYAKE: {np.mean(sensibilidad_no_YAKE_per_fold)} (+- {np.std(sensibilidad_no_YAKE_per_fold)})')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 28.57142984867096%
------------------------------------------------------------------------
> Fold 2 - Sensibilidad YAKE: 88.88888955116272 - Sensibilidad NOYAKE: 9.090909361839294%
------------------------------------------------------------------------
> Fold 3 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 33.33333432674408%
------------------------------------------------------------------------
> Fold 4 - Sensibilidad YAKE: 9.090909361839294 - Sensibilidad NOYAKE: 77.77777910232544%
------------------------------------------------------------------------
> Fold 5 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 0.0%
------------------------------------------------------------------------
> Fold 6 - Sensibilidad YAKE: 0.0 - Sensibilidad NOYAKE: 100.0%
----------